In [1]:
#-- Import crucial libraries. --#
import os
import pandas as pd
import numpy as np

In [4]:
raw_data_path = '../Processed/roads_final_cleaned.csv'
output_data_path = '../Processed/efficiency_stations.csv'

In [6]:
if not os.path.exists(raw_data_path):
    raise FileNotFoundError(f"Cannot file {raw_data_path}")

df = pd.read_csv(raw_data_path)

In [7]:
df['aadt_allve'] = pd.to_numeric(df['aadt_allve'], errors='coerce')
df['last_year'] = pd.to_numeric(df['last_year'], errors='coerce')

In [8]:
df['road_name'] = df['declared_r'].fillna('Unknown Road')
df_clean = df.dropna(subset=['aadt_allve', 'x', 'y']).copy()

print(f"Numbers after cleaning: {len(df_clean)}")

Numbers after cleaning: 57947


In [9]:
invalid_keywords = ['UNKNOWN ROAD', 'UNKNOWN', 'NOT AVAILABLE', 'NAN', 'NONE', '']
df_filtered = df_clean[~df_clean['road_name'].str.upper().isin(invalid_keywords)].copy()

print(f"Number of stations after deleting Unknown: {len(df_filtered)}")

Number of stations after deleting Unknown: 41864


In [10]:
road_averages = df_filtered.groupby('road_name')['aadt_allve'].mean().rename('road_avg_aadt')
df_final = df_filtered.join(road_averages, on='road_name')

df_final['efficiency_index'] = (df_final['aadt_allve'] / df_final['road_avg_aadt']).round(2)

df_final['vs_average_pct'] = ((df_final['efficiency_index'] - 1) * 100).round(1)

print("Calculating finished Efficiency.")

Calculating finished Efficiency.


In [11]:
final_output = df_final.sort_values(['road_name', 'efficiency_index'], 
                                    ascending=[True, False]).groupby('road_name').head(3)

# Lưu file
final_output.to_csv(output_data_path, index=False)

print(f"--- Finished ---")
print(f"File saved at: {output_data_path}")
print(f"Numbers of stations: {len(final_output)}")

final_output[['tfm_id', 'road_name', 'aadt_allve', 'efficiency_index']].head(10)

--- Finished ---
File saved at: ../Processed/efficiency_stations.csv
Numbers of stations: 3504


,tfm_id,road_name,aadt_allve,efficiency_index
43271,42013,160 SOUTH ROADB 2500F,40,1.00
42419,52075,2000F 2070B,13000,2.43
39048,36894,2000F 2070B,8400,1.57
33803,54241,2000F 2070B,0,0.00
30403,9101,2000F 2090B,8600,1.53
40160,47826,2000F 2090B,8200,1.46
41704,47694,2000F 2090B,5700,1.01
41186,48244,2000F 2290B,9100,2.00
34789,20921,2000F 2290B,0,0.00
37217,4652,2000F 2750B,43000,2.33
